In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
# Dataset de ejemplo
sentences = [
    "I love deep learning",
    "I love machine learning",
    "deep learning is fun"
]

In [3]:
# Tokenizar las frases (convertir las palabras a índices)
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

In [7]:
# Vocabulario
vocab_size = len(tokenizer.word_index) + 1  # Añadimos 1 para considerar el índice 0 (pad)
print("Vocabulary size:", vocab_size)
tokenizer.word_index

Vocabulary size: 8


{'learning': 1, 'i': 2, 'love': 3, 'deep': 4, 'machine': 5, 'is': 6, 'fun': 7}

In [5]:

# Convertir las frases a secuencias de índices
sequences = tokenizer.texts_to_sequences(sentences)

In [6]:
# Pad sequences para asegurar que todas las secuencias tengan la misma longitud
max_length = max(len(seq) for seq in sequences)
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

print("Padded sequences:", padded_sequences)

Padded sequences: [[2 3 4 1]
 [2 3 5 1]
 [4 1 6 7]]


# CREAR EL MODELO DE RED NEURONAL PARA EL EMBEDDING

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, GlobalAveragePooling1D

# Definir el modelo
embedding_dim = 8  # Dimensión de los embeddings (cada palabra será representada por un vector de 8 números)

model = Sequential()

# Capa de Embeddings
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length))

# Capa de GlobalAveragePooling1D para hacer un resumen de la secuencia
model.add(GlobalAveragePooling1D())

# Capa densa para clasificación (en este caso, un ejemplo de regresión)
model.add(Dense(1, activation='sigmoid'))

# Compilar el modelo
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Resumen del modelo
model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d             │ ?                           │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# ENTRENAMOS EL MODELO

In [9]:
# Etiquetas de ejemplo para clasificación binaria (por ejemplo, 0 o 1)
labels = [1, 0, 1]  # Aquí se podría tener una etiqueta binaria para cada frase

# Convertir las etiquetas en un tensor
labels_tensor = tf.convert_to_tensor(labels)

# Entrenar el modelo
model.fit(padded_sequences, labels_tensor, epochs=5)

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.6667 - loss: 0.6908
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.6667 - loss: 0.6896
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 1.0000 - loss: 0.6885
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 1.0000 - loss: 0.6874
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 1.0000 - loss: 0.6862


In [10]:
# Obtener los embeddings aprendidos por la capa
embeddings = model.layers[0].get_weights()[0]
print("Embeddings learned by the model:")
print(embeddings)

Embeddings learned by the model:
[[ 0.04583682 -0.04803865 -0.03995817  0.01845526 -0.00979383 -0.02609
  -0.00429933 -0.02665693]
 [ 0.02525458  0.00495477  0.01895732 -0.02507136 -0.04976169  0.00168924
   0.03828722  0.00367266]
 [-0.01321937 -0.01084102  0.04406699 -0.02175142  0.01490192 -0.01549672
  -0.01736899  0.02667702]
 [ 0.04877084 -0.02741879 -0.02390735  0.00384873  0.03115859  0.0425075
  -0.02472027 -0.01579692]
 [-0.02373729  0.02500625 -0.02745852  0.04946198 -0.02346787 -0.03138683
   0.0529897   0.00425453]
 [-0.02993785  0.00464966 -0.01669823 -0.00590976  0.03396635 -0.02311907
   0.04481626  0.00265175]
 [-0.04747578  0.00168185 -0.00145136  0.02298281 -0.00471379  0.00540549
   0.02883075  0.00247099]
 [ 0.0150716  -0.03594451  0.03925766 -0.01339165  0.02632665 -0.01733162
   0.00866954 -0.01941983]]
